# 🤗 Introduction to Hugging Face Transformers



### What We'll Cover in Today's Session :
* **Transformers:** Understand the core idea behind Transformer models
*  **Hugging Face Pipelines:** Perform sentiment analysis, text generation, summarization, translation, and more
* **Under the Hood:** Explore tokenizers and models
* **Hugging Face Hub:** Find, load, and swap between different models
*  **Fine-Tuning:** Train a model on a real-world dataset
*  **Gradio:** Turn your model into a shareable web demo


### Two Tracks
Sections marked **⭐ Core** are the topics we’ll cover together during today’s workshop (~90 minutes).

Sections marked **🎁 Bonus** are optional topics you can explore later at your own pace, or if we have extra time

### Before we start
1. Make sure you're signed into a Google account (for Colab)
2. **File → Save a copy in Drive** so your edits are saved
3. **Turn on a free GPU:** Runtime → Change runtime type → **T4 GPU**. Most sections work fine on CPU, but the fine-tuning section (Section 9) needs a GPU to finish in a few minutes.

## ⭐ 1. What is a Transformer?

**Hugging Face** is a company and open-source community that hosts:
- 🧠 **Models** — pre-trained AI models anyone can use (500,000+ on the Hub!)
- 📊 **Datasets** — ready-to-use datasets for training/testing models
- 🚀 **Spaces** — live demos of ML apps
- 🐍 **`transformers`** — the Python library we'll use today to load and run models in a few lines of code

**Transformers** (the architecture, not the toy robots!) are the neural network design behind almost every major AI model you've heard of — ChatGPT, BERT, Llama, Whisper, Stable Diffusion's text encoder, and more.

The key idea, in plain English:
> A transformer reads a whole sequence (like a sentence) at once and learns which words should "pay attention" to which other words, no matter how far apart they are. That's why they're so good at understanding context.

Example: in *"The trophy didn't fit in the suitcase because **it** was too big"* — a transformer can learn that "it" refers to the trophy, not the suitcase, by attending to the right context.



## ⭐ 2. Setup

Let's install the `transformers` library (Colab has most dependencies already, this just makes sure we have the latest).


In [1]:
!pip install -q transformers datasets accelerate gradio torch
print("Setup Complete")

Setup Complete


In [2]:
import torch
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cpu
GPU available: False


> 💡 It's  fine if `GPU available: False` — everything in this workshop runs on CPU too, just a little slower.


## ⭐ 3. Your First Pipeline: Sentiment Analysis

The `pipeline()` function is the easiest way to use a Hugging Face model. You give it a **task name**, and it automatically:
1. Downloads a sensible pre-trained model for that task
2. Downloads the matching tokenizer
3. Wraps them so you can just pass in text and get an answer


In [ ]:
from transformers import pipeline

# Create a sentiment-analysis pipeline
classifier = pipeline("sentiment-analysis")

# Try it out!
result = classifier("I absolutely loved this workshop!")
print(result)


**What just happened?**
- `pipeline("sentiment-analysis")` downloaded a pre-trained model (by default, a DistilBERT model fine-tuned on sentiment)
- It tokenized your sentence, ran it through the model, and converted the output into a human-readable label + confidence score


### 🙋 Try it yourself
Run the cell below and replace the text with your own sentences. Try something ambiguous or sarcastic — does it get it right?

In [ ]:
my_sentences = [
    "I can't believe how good this pizza is.",
    "This is the worst customer service I've ever experienced.",
    "I guess it was okay, nothing special.",
    # add your own sentence here!
]

results = classifier(my_sentences)
for sentence, result in zip(my_sentences, results):
    print(f"{sentence}\n  → {result['label']} (confidence: {result['score']:.2f})\n")


#### **Pipeline options worth knowing**

Pipelines accept parameters that change their behavior. For example, `top_k=None` returns the score for **every** label, not just the winner. That's useful for seeing how confident the model really is.


In [ ]:
classifier("The movie was fine, I guess.", top_k=None)

## ⭐ 4. A Tour of Pipelines

`pipeline()` supports dozens of tasks. Let's try a few of the most useful ones. For each, we'll run a demo, then you try it with your own input.

### 4.1 Text Generation
Give the model a prompt, and it predicts what comes next.


In [ ]:
generator = pipeline("text-generation", model="gpt2")

output = generator(
    "In the future, artificial intelligence will",
    max_length=40,
    num_return_sequences=1
)
print(output[0]["generated_text"])



🙋 **Try it:** Change the prompt below to something fun and re-run.

In [ ]:
my_prompt = "The best part about being a computer science student is"

output = generator(my_prompt, max_length=40, num_return_sequences=1)
print(output[0]["generated_text"])


### 4.2 Summarization
Feed in a long passage, get a short summary back.


In [ ]:
summarizer = pipeline("summarization")

article = """
Hugging Face is a company that has become a central hub for the machine learning
community. It provides an open-source library called Transformers, which gives
developers easy access to thousands of pre-trained models for tasks like text
classification, translation, summarization, and question answering. Beyond the
library, Hugging Face also hosts the Hugging Face Hub, a platform where anyone
can share models, datasets, and interactive demos called Spaces. This has made
state-of-the-art AI far more accessible to students, researchers, and hobbyists
who don't have the resources to train massive models from scratch.
"""

summary = summarizer(article, max_length=60, min_length=20, do_sample=False)
print(summary[0]["summary_text"])




### 4.3 Zero-Shot Classification
This is a favorite — the model can classify text into categories **it was never explicitly trained on**, just by describing the categories in plain English.


In [ ]:
zero_shot = pipeline("zero-shot-classification")

text = "I need to fix a bug in my Python code before the deadline tonight."
candidate_labels = ["technology", "cooking", "sports", "politics"]

result = zero_shot(text, candidate_labels)
for label, score in zip(result["labels"], result["scores"]):
    print(f"{label}: {score:.2f}")


🙋 **Try it:** Change `text` and `candidate_labels` below to your own example.


In [ ]:
text = "Replace this with your own sentence."
candidate_labels = ["label1", "label2", "label3"]

result = zero_shot(text, candidate_labels)
for label, score in zip(result["labels"], result["scores"]):
    print(f"{label}: {score:.2f}")


## ⭐ 5. Under the Hood, Part 1: Tokenizers

Models don't understand words — they understand **numbers**. A **tokenizer** converts text into numbers (and back).

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

text = "Transformers are amazing!"
tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)

input_ids = tokenizer.convert_tokens_to_ids(tokens)
print("Token IDs:", input_ids)

# Or do both steps (plus special tokens) at once:
encoded = tokenizer(text)
print("\nFull encoding:", encoded)



**Notice:**
- Words get split into smaller pieces (subwords) — this is how the model handles words it's never seen before
- `input_ids` are the numbers the model actually processes
- `attention_mask` tells the model which tokens are real vs. padding (useful when batching sentences of different lengths)

🙋 **Try it:** Tokenize your own sentence, including something long or unusual (like your name, or a made-up word).


In [ ]:
your_text = "Replace this with your own text!"
print(tokenizer(your_text))

## ⭐ 6. Under the Hood, Part 2: Models

Now let's load the actual model and run the full process manually — this is exactly what `pipeline()` was doing for us automatically.


In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)

text = "I really enjoyed learning about transformers today!"

# Step 1: Tokenize (convert text -> numbers), return PyTorch tensors
inputs = tokenizer(text, return_tensors="pt")
print("Model input:", inputs)

# Step 2: Run the model (no gradient tracking needed, we're not training)
with torch.no_grad():
    outputs = model(**inputs)

print("\nRaw output (logits):", outputs.logits)



Those raw numbers ("logits") aren't probabilities yet. We convert them using **softmax**:


In [ ]:
import torch.nn.functional as F

probabilities = F.softmax(outputs.logits, dim=-1)
print("Probabilities:", probabilities)

# Map back to human-readable labels
labels = model.config.id2label
for i, prob in enumerate(probabilities[0]):
    print(f"{labels[i]}: {prob.item():.2%}")



🎉 **You just reproduced what `pipeline("sentiment-analysis")` does automatically, step by step:**

1. Tokenizer: text → numbers
2. Model: numbers → raw logits
3. Softmax: logits → probabilities
4. Label mapping: probabilities → human-readable output


## ⭐ 7. Exploring the Hugging Face Hub

Every pipeline we used picked a **default model**. But the [Hugging Face Hub](https://huggingface.co/models) has 500,000+ models you can swap in — different languages, sizes, specialties, and licenses.

Let's try swapping the sentiment model for one trained on **multilingual** tweets.


In [ ]:
multilingual_sentiment = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

examples = [
    "Me encanta este taller!",       # Spanish
    "Ce cours est vraiment nul.",    # French
    "This workshop is fantastic!",   # English
]

for text in examples:
    print(text, "→", multilingual_sentiment(text))



### How to find models yourself
1. Go to **huggingface.co/models**
2. Filter by task (left sidebar) — e.g. "Text Classification", "Translation"
3. Sort by "Most downloads" or "Trending" to find well-tested options
4. Each model has a **model card** — read it! It tells you what data it was trained on, its limitations, and how to use it
5. Copy the model name (e.g. `cardiffnlp/twitter-xlm-roberta-base-sentiment`) and drop it into `pipeline("task", model="...")`

🙋 **Try it:** Go to the Hub, find a different sentiment or text-classification model, and swap it in below.


In [ ]:
# Paste a model name from huggingface.co/models here
my_model_name = "PASTE_MODEL_NAME_HERE"

# Uncomment once you've picked a model:
# custom_pipeline = pipeline("sentiment-analysis", model=my_model_name)
# print(custom_pipeline("Try a sentence here!"))


## ⭐ 8. Datasets Library

Models learn from data. Hugging Face hosts thousands of datasets you can load in one line. We'll use **Rotten Tomatoes**, a dataset of movie review snippets labeled positive (1) or negative (0).


In [ ]:
from datasets import load_dataset

dataset = load_dataset("rotten_tomatoes")
print(dataset)

In [ ]:
# Look at a few examples
for i in range(3):
    example = dataset["train"][i]
    label = "positive" if example["label"] == 1 else "negative"
    print(f"[{label}] {example['text']}\n")


**What to notice:**
- The dataset is already split into `train`, `validation`, and `test`
- The **train** split is what the model learns from. The **test** split is held back to check how well it learned. It's like studying with practice problems vs. taking the real exam.
- Each example is a dictionary with a `text` and a `label`

🙋 **Try it:** How many examples are in the training split? (Hint: `len(dataset["train"])`)


## ⭐ 9. Fine-Tuning Your Own Model

So far we've used models exactly as someone else trained them.

**Fine-tuning** means taking a pre-trained model and training it a little more on *your* data so it gets better at *your* task.


> ⚠️ **Make sure your runtime is set to a T4 GPU** (Runtime → Change runtime type). Training on CPU will take much too long.

We'll fine-tune **DistilBERT** (a small, fast transformer) on a small slice of Rotten Tomatoes. The steps:
1. Pick a small subset of data (so training takes a couple of minutes)
2. Tokenize it
3. Load a fresh model
4. Measure accuracy **before** training
5. Train
6. Measure accuracy **after** training


### Step 1: Prepare a small subset


In [ ]:
train_data = dataset["train"].shuffle(seed=42).select(range(2000))
eval_data = dataset["test"].shuffle(seed=42).select(range(500))
print(len(train_data), "training examples,", len(eval_data), "evaluation examples")


### Step 2: Tokenize
We apply the tokenizer to every example. `batched=True` makes it fast.


In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_data.map(tokenize, batched=True)
tokenized_eval = eval_data.map(tokenize, batched=True)
print(tokenized_train[0].keys())


### Step 3: Load a fresh model
We use `num_labels=2` because we have two classes (negative/positive). You'll see a warning that some weights are newly initialized. **That's expected!** The classification head on top is brand new and hasn't been trained yet.


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

### Step 4: Set up the Trainer
The `Trainer` class handles the training loop (batching, backpropagation, evaluation) so you don't have to write it yourself. We also define a simple **accuracy** metric.


In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": (predictions == labels).mean()}

training_args = TrainingArguments(
    output_dir="my-finetuned-model",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=25,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)


### Step 5: Accuracy BEFORE training
The model hasn't learned this task yet, so we expect roughly 50% (a coin flip).


In [ ]:
before = trainer.evaluate()
print(f"Accuracy before training: {before['eval_accuracy']:.2%}")

### Step 6: Train
This takes a few minutes on a T4 GPU. Watch the **loss** go down. That's the model learning.


In [ ]:
trainer.train()

### Step 7: Accuracy AFTER training


In [ ]:
after = trainer.evaluate()
print(f"Accuracy before: {before['eval_accuracy']:.2%}")
print(f"Accuracy after:  {after['eval_accuracy']:.2%}")

🎉 **You just fine-tuned a transformer.** With only 2,000 examples and a couple of minutes of training, accuracy should jump substantially. Let's try our new model on some sentences.


In [ ]:
from transformers import pipeline

my_pipeline = pipeline(
    "sentiment-analysis",
    model=trainer.model,
    tokenizer=tokenizer,
    device=trainer.model.device,
)

for text in ["A moving, beautifully acted film.", "Dull, predictable, and far too long."]:
    print(text, "->", my_pipeline(text))

> Note: the labels show up as `LABEL_0` / `LABEL_1`. That means 0 = negative and 1 = positive (as we saw in the dataset). You can set custom names via `id2label` when loading the model.

**Where would you go from here?** More data, more epochs, a bigger model, and sharing your model to the Hub with `trainer.push_to_hub()`.


## ⭐ 10. Build a Web Demo with Gradio

A model in a notebook is great for you, but what if you want to show a friend or put it in your portfolio? **Gradio** turns a Python function into a web app in a few lines. This is also what powers Hugging Face **Spaces**.


In [ ]:
import gradio as gr

def analyze_sentiment(text):
    result = my_pipeline(text)[0]
    label = "Positive" if result["label"] == "LABEL_1" else "Negative"
    return {label: result["score"]}

demo = gr.Interface(
    fn=analyze_sentiment,
    inputs=gr.Textbox(lines=3, placeholder="Type a movie review..."),
    outputs=gr.Label(),
    title=" Movie Review Sentiment",
    description="A DistilBERT model fine-tuned by YOU at the ACM-W workshop!",
)

demo.launch(share=True)


`share=True` gives you a **public link** (valid for ~72 hours) that anyone can open, so send it to a friend!

### Making it permanent
To host a demo for free forever:
1. Create an account at [huggingface.co](https://huggingface.co)
2. Create a new **Space** and choose the Gradio SDK
3. Upload your `app.py` and a `requirements.txt`

🙋 **Try it:** Edit the `title` and `description`, add an example input using `examples=[...]`, and re-run.


## 12. 🎁 Bonus: Beyond Text *(~10 min)*

Transformers aren't just for text! The same `pipeline()` idea works for images, audio, and more.

### Image classification


In [ ]:
image_classifier = pipeline("image-classification")

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
predictions = image_classifier(url)
for p in predictions[:3]:
    print(f"{p['label']}: {p['score']:.2f}")


### Zero-shot image classification with CLIP
Just like zero-shot text classification, but for images. You pick the labels in plain English.


In [ ]:
clip = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")

result = clip(url, candidate_labels=["a cat", "a dog", "a car", "a plate of food"])
for r in result:
    print(f"{r['label']}: {r['score']:.2f}")


🙋 **Try it:** Find any image URL online (or upload a file to Colab) and try your own labels.

Other tasks to explore: `automatic-speech-recognition` (audio → text), `image-to-text` (captioning), `object-detection`. Browse them all at [huggingface.co/tasks](https://huggingface.co/tasks).


## 13. 🏆 Mini Challenge *(🎁 Bonus / take-home, ~10 min)*

Pick **one** and build it using what you learned today. Work with a neighbor if you'd like!

**Option A — Mood Detector:** Build a pipeline that takes 3 sentences about your day and prints the overall sentiment for each.

**Option B — Mini Chatbot Opener:** Use `text-generation` to auto-complete 3 different conversation starters.

**Option C — Topic Sorter:** Use `zero-shot-classification` to sort 5 sentences (make them up!) into categories of your choice (e.g. "school", "food", "sports", "technology").

**Option D — Go Multilingual:** Find a translation model on the Hub for a language pair you're interested in, and translate a paragraph.

**Option E — Ship It:** Build a Gradio demo around ANY pipeline from today (summarizer, zero-shot classifier, QA...) and share the link.

Use the empty cell below to build your solution.


In [ ]:
# Your mini challenge code here!


## 14. Wrap-up & Where to Go Next

### Next steps
- 📚 **[Hugging Face NLP Course](https://huggingface.co/learn/nlp-course)** — free, hands-on, goes from here through fine-tuning your own models
- 🎨 **[Hugging Face Spaces](https://huggingface.co/spaces)** — browse and build live ML demos, no server needed
- 🧪 **Go deeper on fine-tuning** — try a different dataset (e.g. `emotion`, `ag_news`), a bigger model, or more epochs, then `push_to_hub()` your result
- 🗂️ **[Datasets library](https://huggingface.co/docs/datasets)** — pairs with `transformers` for loading training data
- 💬 Join our ACM-W community to keep building!

---
### Thank you for coming! 💜
*Questions? Find us after the workshop or reach out via ACM-W.*